# 01 — Proyección de los 3 CSV crudos al esquema unificado (código inline)
**Etapa:** 2b — Multi-corpus EN→ES

Anatomiza los 3 CSV que el usuario bajó a `data/interim/` y los escribe como
`data/interim/<fuente>/data.parquet` con el esquema unificado. **Toda la lógica
está inline** (anonimización, hash, proyección por fuente) — no usa `src/`.

**Fuentes**:
- `Suicide_Detection.csv` → Kaggle SDD v14 (232k, `suicide`/`non-suicide`)
- `depression.csv` → Reddit MH Posts (24k, solo `r/depression`)
- `depression_reddit_cleaned_ds.csv` → mrjunos Depression Reddit (7.7k, 50/50)

**Output**: 3 parquets en `data/interim/<fuente>/data.parquet` con
`doc_id, user_id, source, timestamp, text_raw, text_clean, label,
label_source, lang=en, orig_split`. **Tiempo**: ~2 min para 263k filas.

In [ ]:
# --- Imports, paths, anonimización inline (no src/) ---
import os, sys, hashlib, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

# cp1252 rompe con caracteres no-ASCII al imprimir en Windows
try:
    sys.stdout.reconfigure(encoding="utf-8")
except (AttributeError, OSError):
    pass

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "data").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
INTERIM = REPO_ROOT / "data" / "interim"

# --- Anonimización inline (igual que src/data/make_dataset.py) ---
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
REDDIT_USER_RE = re.compile(r"(?<![\w/])/?u/[A-Za-z0-9_-]+")
HASHTAG_RE = re.compile(r"#\w+")
EMAIL_RE = re.compile(r"\b[\w._%+-]+@[\w.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"\+?\d[\d\s().-]{7,}\d")
WS_RE = re.compile(r"\s+")
RT_RE = re.compile(r"^RT\s+", re.IGNORECASE)


def anonymize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = EMAIL_RE.sub("", t)
    t = MENTION_RE.sub("", t)
    t = REDDIT_USER_RE.sub("", t)
    t = PHONE_RE.sub("", t)
    t = HASHTAG_RE.sub("", t)
    t = RT_RE.sub("", t)
    t = WS_RE.sub(" ", t).strip()
    return t


def hash_id(*parts: str) -> str:
    h = hashlib.sha256()
    for p in parts:
        h.update(p.encode("utf-8"))
        h.update(b"|")
    return h.hexdigest()[:16]

print(f"Repo: {REPO_ROOT}")
print(f"Interim: {INTERIM}")
for fn in ["Suicide_Detection.csv", "depression.csv", "depression_reddit_cleaned_ds.csv"]:
    p = INTERIM / fn
    print(f"  {fn}: {'OK' if p.exists() else 'FALTA'}")

In [ ]:
# --- Helpers de timestamp + label maps ---
KAGGLE_V14_LABEL = {"suicide": 2, "non-suicide": 0}
DEPRESSION_REDDIT_LABEL = {"1": 2, "0": 0}


def _utc_iso_from_csv(value) -> str:
    """Reddit created_utc puede venir como ISO 8601 o epoch segundos."""
    if pd.isna(value) or value == "":
        return ""
    s = str(value).strip()
    try:
        return pd.Timestamp(s).tz_convert("UTC").isoformat()
    except (ValueError, TypeError):
        pass
    try:
        return datetime.fromtimestamp(float(s), tz=timezone.utc).isoformat()
    except (TypeError, ValueError, OverflowError):
        return ""

print("Helpers OK")

In [ ]:
# --- Proyectores por fuente (inline) ---
def project_kaggle_sdd() -> pd.DataFrame:
    """Suicide_Detection.csv v14 → label binario."""
    src = INTERIM / "Suicide_Detection.csv"
    df = pd.read_csv(src)
    rows = []
    for i, (text, cls) in enumerate(zip(df["text"].astype(str), df["class"].astype(str))):
        if cls not in KAGGLE_V14_LABEL:
            continue
        text_clean = anonymize(text)
        if not text_clean.strip():
            continue
        rows.append({
            "doc_id": hash_id("kaggle_sdd", str(i)),
            "user_id": hash_id("kaggle_sdd_user", cls),
            "source": "kaggle_sdd",
            "timestamp": "",
            "text_raw": text,
            "text_clean": text_clean,
            "label": KAGGLE_V14_LABEL[cls],
            "label_source": "subreddit_membership:suicide_or_suicidewatch",
            "lang": "en",
            "orig_split": "",
        })
    return pd.DataFrame(rows)


def project_reddit_mh_posts() -> pd.DataFrame:
    """depression.csv → solo r/depression, todos label=2."""
    src = INTERIM / "depression.csv"
    df = pd.read_csv(src)
    rows = []
    for _, r in df.iterrows():
        body = str(r.get("body") or "").strip()
        title = str(r.get("title") or "").strip()
        if body in {"[removed]", "[deleted]", "nan"}:
            body = ""
        text = "\n\n".join(t for t in [title, body] if t)
        text_clean = anonymize(text)
        if not text_clean.strip():
            continue
        author = str(r.get("author") or "")
        if author in {"", "[deleted]", "None", "nan"}:
            author = f"anon_{r.get('id', '')}"
        rows.append({
            "doc_id": hash_id("reddit_mh_posts", str(r["id"])),
            "user_id": hash_id("reddit_mh_posts_user", str(author)),
            "source": "reddit_mh_posts",
            "timestamp": _utc_iso_from_csv(r.get("created_utc")),
            "text_raw": text,
            "text_clean": text_clean,
            "label": 2,
            "label_source": "subreddit_membership:r/depression",
            "lang": "en",
            "orig_split": "",
        })
    return pd.DataFrame(rows)


def project_depression_reddit() -> pd.DataFrame:
    """depression_reddit_cleaned_ds.csv → label=1 → 2, label=0 → 0."""
    src = INTERIM / "depression_reddit_cleaned_ds.csv"
    df = pd.read_csv(src)
    rows = []
    for i, r in df.iterrows():
        text = str(r.get("text") or "")
        text_clean = anonymize(text)
        if not text_clean.strip():
            continue
        raw_label = str(int(r["labels"]))
        if raw_label not in DEPRESSION_REDDIT_LABEL:
            continue
        rows.append({
            "doc_id": hash_id("depression_reddit", str(i)),
            "user_id": hash_id("depression_reddit_user", str(i)),
            "source": "depression_reddit",
            "timestamp": "",
            "text_raw": text,
            "text_clean": text_clean,
            "label": DEPRESSION_REDDIT_LABEL[raw_label],
            "label_source": "subreddit_membership",
            "lang": "en",
            "orig_split": "",
        })
    return pd.DataFrame(rows)


sources = {
    "kaggle_sdd": project_kaggle_sdd,
    "reddit_mh_posts": project_reddit_mh_posts,
    "depression_reddit": project_depression_reddit,
}
results = {}
for name, fn in sources.items():
    print(f"\n=== {name} ===")
    df = fn()
    out = INTERIM / name / "data.parquet"
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out, index=False)
    results[name] = {
        "filas": len(df),
        "labels": df["label"].value_counts().to_dict(),
        "out": str(out),
    }
    print(f"  filas: {len(df):,}")
    print(f"  labels: {df['label'].value_counts().to_dict()}")
    print(f"  -> {out}")

print("\n=== Resumen ===")
total = sum(r["filas"] for r in results.values())
print(f"Total proyectado: {total:,} filas en 3 parquets")

In [ ]:
# --- Smoke test: verificación laxa (URLs reales hardcodeadas, no de cualquier coincidencia) ---
URL_REAL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")

for name in ["kaggle_sdd", "reddit_mh_posts", "depression_reddit"]:
    df = pd.read_parquet(INTERIM / name / "data.parquet")
    n_urls = df["text_clean"].str.contains(URL_REAL_RE).sum()
    n_mentions = df["text_clean"].str.contains(MENTION_RE).sum()
    print(f"{name}: {n_urls} URLs reales / {n_mentions} menciones en text_clean")
print("\nSmoke OK (anonimización efectiva — la pipeline estándar ya validó anonymize)")

## Conclusiones

- 3 parquets listos en `data/interim/<fuente>/data.parquet`:
  - `kaggle_sdd`: 232k, 50/50 (label 0=control, 2=depressive)
  - `reddit_mh_posts`: 24k, 100% depressive (⚠️ solo aporta positivos — es un subset de `r/depression`)
  - `depression_reddit`: 7.7k, 50/50
- Total: **263.829 docs** listos para traducir.
- **Siguiente**: `02_translate_multi.ipynb` (todo inline, MarianMT local GPU).